# Agent Orchestration

## Northstar: a durable incident proposal workflow

At 09:04 checkout conversion falls in Europe. Metrics, logs, and deployment history must be collected; an agent may synthesize a mitigation proposal, but it may not decide graph transitions, grant itself access, or execute a remediation.

**Outcomes.** Distinguish intelligence from orchestration; model a state machine and parallel DAG; use events, queues, scheduling, checkpoints, approval, recovery, and long-running budgets; and select a framework by durability needs. **Safety boundary:** all code is deterministic and creates no external side effects.

![Agent orchestration graph](../../../assets/agent-orchestration.svg)

The diagram is a static SVG so it renders reliably in GitHub and local Jupyter. The orange boundary is important: the agent reasons only inside an approved node; application code owns state, authorization, retries, budgets, and action execution.

## Step 1 — write the durable run contract

A production run needs a stable identifier, tenant and owner, deadline, cancellation status, route, policy version, evidence references, action fingerprint, retry count, budget, idempotency keys, and terminal reason. Persisting an LLM conversation alone cannot recover correct business state.

This lab uses an explicit state machine. The transition table is deliberately smaller than a production graph, making it easy to inspect: `route → parallel-evidence → approval → complete`, with `cancelled` and `escalated` terminal states.

In [ ]:
"""Credential-free, deterministic orchestration controls for the Northstar case.

The model may synthesize a proposal, but it never advances durable state or grants
itself authority.  Those transitions belong to this application-owned controller.
"""
from dataclasses import dataclass, field


TERMINAL = {"complete", "cancelled", "escalated"}


@dataclass
class Run:
    run_id: str = "inc-eu-104"
    state: str = "route"
    trace: list[str] = field(default_factory=list)
    evidence: set[str] = field(default_factory=set)
    approved: bool = False
    attempts: int = 0
    budget_remaining: int = 4
    action_fingerprint: str = "proposal:rollback:checkout:deploy-842"
    processed_events: set[str] = field(default_factory=set)


def record(run: Run, item: str) -> None:
    run.trace.append(f"{run.state}:{item}")


def step(run: Run, event: str = "", event_id: str = "") -> str:
    """Advance one safe state transition; duplicate events have no side effect."""
    if event_id and event_id in run.processed_events:
        record(run, f"duplicate-event:{event_id}")
        return run.state
    if event_id:
        run.processed_events.add(event_id)
    if run.state in TERMINAL:
        record(run, "terminal-noop")
    elif event == "cancel":
        run.state = "cancelled"
        record(run, "cancelled")
    elif run.budget_remaining <= 0:
        run.state = "escalated"
        record(run, "budget-exhausted")
    elif run.state == "route":
        run.state = "parallel-evidence"
        run.budget_remaining -= 1
        record(run, "route:bounded-agent")
    elif run.state == "parallel-evidence":
        if event in {"metrics", "logs", "deployments"}:
            run.evidence.add(event)
            record(run, f"evidence:{event}")
        if {"metrics", "logs", "deployments"} <= run.evidence:
            run.state = "approval"
            record(run, "join:proposal-checkpointed")
    elif run.state == "approval" and event == "approve":
        run.approved = True
        run.state = "complete"
        record(run, f"approval:{run.action_fingerprint}")
    elif run.state == "approval" and event in {"reject", "expired"}:
        run.state = "escalated"
        record(run, f"approval-{event}")
    elif run.state == "approval":
        record(run, "checkpoint:waiting-approval")
    return run.state


def retry_read(run: Run, error: str) -> str:
    """Retry only bounded, idempotent reads; escalate all other failures."""
    run.attempts += 1
    if error == "timeout" and run.attempts <= 2:
        record(run, f"retry-read:{run.attempts}")
        return "retry"
    run.state = "escalated"
    record(run, f"escalate:{error}")
    return "escalate"


def run_demo() -> Run:
    run = Run()
    step(run)
    for source in ("metrics", "logs", "deployments"):
        step(run, source, f"evidence-{source}")
    assert run.state == "approval"
    step(run, "approve", "approval-1")
    step(run, "approve", "approval-1")
    assert run.state == "complete" and run.approved
    return run


if __name__ == "__main__":
    print(run_demo())


In [1]:
from pathlib import Path
if not (TOPIC / 'lab.py').exists():

run = Run()
print(run)
assert run.state == 'route' and run.budget_remaining == 4

Run(run_id='inc-eu-104', state='route', trace=[], evidence=set(), approved=False, attempts=0, budget_remaining=4, action_fingerprint='proposal:rollback:checkout:deploy-842', processed_events=set())


## Step 2 — route, then execute independent reads in parallel

Routing is deterministic: known low-risk status questions use a workflow; ambiguous incidents enter this bounded investigation route. A DAG represents dependency-ready tasks *within* the state machine. Metrics, logs, and deployments are read-only and independent, so a real orchestrator may dispatch them concurrently with a concurrency limit. The join is a contract: no proposal before all required evidence arrives.

In [2]:
step(run)  # route selects the bounded-agent investigation path
for source in ('metrics', 'logs', 'deployments'):
    step(run, source, f'evidence-{source}')

print(run.state, sorted(run.evidence))
print(*run.trace, sep='\n')
assert run.state == 'approval'
assert run.evidence == {'metrics', 'logs', 'deployments'}

approval ['deployments', 'logs', 'metrics']
parallel-evidence:route:bounded-agent
parallel-evidence:evidence:metrics
parallel-evidence:evidence:logs
parallel-evidence:evidence:deployments
approval:join:proposal-checkpointed


## Step 3 — checkpoint and human approval

The approval node stores the exact `action_fingerprint`, rather than a broad statement such as *fix checkout*. An approval event must come from an authenticated source and match the proposal, tenant, policy version, expiry, budget, and cancellation state. Duplicate delivery is expected in queues and event brokers, so the `event_id` makes the transition idempotent. Approval is permission to evaluate a specific action at the boundary—not a standing credential.

In [3]:
step(run, 'approve', 'approval-eu-1')
step(run, 'approve', 'approval-eu-1')  # duplicate queue delivery: no second side effect
print(run.state, run.approved)
print(*run.trace[-2:], sep='\n')
assert run.state == 'complete' and run.approved
assert any('duplicate-event' in item for item in run.trace)

complete True
complete:approval:proposal:rollback:checkout:deploy-842
complete:duplicate-event:approval-eu-1


## Step 4 — recovery, retries, long-running work, and scheduling

Retry only idempotent reads, with a bounded attempt count and backoff in a real service. If an action request times out, reconcile via its idempotency key; do not blindly replay it. A scheduled job should create or wake a bounded run with an owner, lease/heartbeat, deadline, and cancellation mechanism. Event-driven wakeups (approval, ticket update, deploy webhook) reduce polling and preserve an auditable causal link.

The next experiment produces a retry for a transient read timeout, then escalates after the retry budget is exhausted.

In [4]:
recovery = Run(state='parallel-evidence')
assert retry_read(recovery, 'timeout') == 'retry'
assert retry_read(recovery, 'timeout') == 'retry'
assert retry_read(recovery, 'timeout') == 'escalate'
print(recovery.state)
print(*recovery.trace, sep='\n')
assert recovery.state == 'escalated'

# Failure case: a cancelled run is terminal and cannot be revived by a late approval.
cancelled = Run(state='approval')
step(cancelled, 'cancel', 'cancel-1')
step(cancelled, 'approve', 'late-approval')
assert cancelled.state == 'cancelled'

escalated
parallel-evidence:retry-read:1
parallel-evidence:retry-read:2
escalated:escalate:timeout


## Framework choice and production judgment

- Use **LangGraph** for explicit agent state graphs, persistence, interrupts, and conditional edges.
- Use **Temporal** when work must survive worker/process failures for minutes, hours, or days with timers, signals, retries, and replay.
- Use **Prefect, Dagster, or Airflow** for data-heavy scheduled DAGs.
- Use **OpenAI Agents SDK, CrewAI Flows, or AutoGen** to manage bounded agent/tool/team behavior, while retaining application-owned authorization and durable workflow controls.

**Exercises:** (1) add an `expired` timer for approval; (2) add a `policy-version-mismatch` transition; (3) allow only two parallel reads; (4) store a redacted trace event; (5) compare this state machine to a Temporal workflow and explain what must remain deterministic.

References: [LangGraph durable execution](https://docs.langchain.com/oss/python/langgraph/durable-execution), [Temporal workflows](https://docs.temporal.io/workflows), [OpenAI practical guide](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/), [CrewAI Flows](https://docs.crewai.com/en/concepts/flows).